# Exercício 11 — Regressão com scikit-learn

Este exercício repete o pipeline da aula na **sua coleta de rede social** (a mesma que você usou nas Aulas 5 a 7). Você vai treinar um modelo que estima a taxa de engajamento de um post a partir de características que existem *antes* da publicação, e comparar esse modelo com um chute burro.

Não é para inventar coleta nova. Copie a sua `exportacao.csv` das aulas anteriores para `dados/exportacao.csv` dentro desta pasta.

Antes de tudo, copie a pasta `exercicios/` para dentro da sua pasta de entregas (`extracao-dados-trabalhos-seunome`, com o SEU nome), numa pasta `11-regressao-scikit-learn` dentro de `projetos/`.

## Preparação do ambiente

Este notebook mora numa pasta própria (dentro da sua pasta de entregas). Crie o ambiente e instale as dependências dentro dela.

No Windows (Prompt de Comando ou Terminal integrado do VS Code):

```cmd
uv venv .venv
uv pip install -r requirements.txt
```

No Mac (Terminal), os mesmos comandos. Se o `uv` não funcionar, use `pip install -r requirements.txt` com o ambiente já ativado.

## Parte 0 — Dados

**Fonte dos dados:** copie para `dados/exportacao.csv` a mesma coleta que você usou nas Aulas 5 a 7.

**Pergunta que este modelo tenta responder (preencha):**

> Dá para estimar a taxa de engajamento de um post da minha coleta a partir de características do autor, do tamanho da legenda, do número de hashtags e do horário?

## Parte 1 — Carregar e construir o alvo

Esse trecho já vem pronto (é o mesmo da aula). Ajuste os nomes de coluna se a sua plataforma usar nomes diferentes de `likes`, `comments`, `shares`, `plays`.

In [6]:
import pandas as pd
df = pd.read_csv("/Users/leonardorosa/estudo/extracao-dados-trabalhos-juliacereja/projetos/11-regressao-scikit-learn/dados/exportacao.csv", sep=";")
print(df.columns)

Index(['collected_from_url', 'id', 'thread_id', 'author', 'author_full',
       'author_followers', 'author_likes', 'author_videos', 'author_avatar',
       'body', 'stickers', 'timestamp', 'unix_timestamp', 'is_duet', 'is_ad',
       'is_paid_partnership', 'is_sensitive', 'is_photosensitive',
       'music_name', 'music_id', 'music_url', 'music_thumbnail',
       'music_author', 'video_url', 'tiktok_url', 'thumbnail_url', 'likes',
       'comments', 'shares', 'plays', 'hashtags', 'challenges',
       'diversification_labels', 'location_created', 'effects', 'warning'],
      dtype='str')


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/Users/leonardorosa/estudo/extracao-dados-trabalhos-juliacereja/projetos/11-regressao-scikit-learn/dados/exportacao.csv", sep=";")
df = df.drop_duplicates() #elimina linhas duplicadas
df = df[df["plays"] > 0].copy() #elimina linhas com zero plays

df["taxa_engajamento"] = (
    df["likes"] + df["comments"] + df["shares"]
) / df["plays"]

y = df["taxa_engajamento"] #variável alvo
print(f"Posts na base: {len(df)}")
y.describe() #estatísticas descritivas da variável alvo

Posts na base: 1416


count    1416.000000
mean        0.064734
std         0.056962
min         0.000000
25%         0.018917
50%         0.049304
75%         0.095114
max         0.402174
Name: taxa_engajamento, dtype: float64

>std = 0.0570 → é o desvio padrão, ou seja, o quanto os valores costumam variar em torno da média. Um std relativamente alto perto da média (quase do mesmo tamanho dela) já é um sinal de que os dados são bem dispersos — tem posts bem diferentes uns dos outros em termos de engajamento.

>max = 0.4022 → o post com melhor desempenho teve 40,2% de engajamento — bem acima da média, o que sugere que esse (ou esses) posts foram outliers de sucesso.

>Um ponto importante pra sua análise: repare que a média (6,47%) é maior que a mediana (4,93%). Isso é um indício clássico de distribuição assimétrica à direita (right-skewed): a maioria dos posts tem engajamento baixo, mas alguns poucos posts com engajamento muito alto (como aquele de 40%) estão "puxando" a média pra cima. Se a distribuição fosse simétrica, média e mediana seriam praticamente iguais.

## Parte 2 — Construir as suas features

**Regra de ouro: nada de vazamento.** `likes`, `comments`, `shares` e `plays` NÃO entram no `X` — eles são o alvo.

Monte um `X` com **pelo menos três features derivadas por você**. Pode reaproveitar as da aula (nº de hashtags, tamanho da legenda, nº de emojis, hora, dia da semana, seguidores do autor) e/ou inventar outras que façam sentido para a sua coleta (tem menção `@`? tem link? tem ponto de interrogação? é fim de semana?).

In [13]:
import re

X = pd.DataFrame(index=df.index)

# complete aqui: crie pelo menos TRÊS colunas de feature em X.
# exemplos da aula (use, troque ou acrescente):

X["seguidores_autor"] = df["author_followers"]
X["tam_legenda"] = df["body"].fillna("").str.len()
X["n_hashtags"] = df["hashtags"].fillna("").apply(lambda s: 0 if s == "" else len(s.split(",")))
momento = pd.to_datetime(df["timestamp"])
X["hora"] = momento.dt.hour



print("Features criadas:", list(X.columns))
X.head()

Features criadas: ['seguidores_autor', 'tam_legenda', 'n_hashtags', 'hora']


,seguidores_autor,tam_legenda,n_hashtags,hora
0,3000000,66,5,20
1,16300000,94,4,22
2,53700,78,3,12
3,12100,65,5,12
4,9503,106,5,17


**Confira antes de seguir:** todas as colunas de `X` são numéricas? Nenhuma é `likes`/`comments`/`shares`/`plays`? Nenhuma tem valor ausente? (`X.isna().sum()` e `X.dtypes`)

In [14]:
print(X.dtypes)
print()
print("valores ausentes por coluna:")
print(X.isna().sum())

seguidores_autor    int64
tam_legenda         int64
n_hashtags          int64
hora                int32
dtype: object

valores ausentes por coluna:
seguidores_autor    0
tam_legenda         0
n_hashtags          0
hora                0
dtype: int64


## Parte 3 — Treino, teste e modelo bobo

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# modelo bobo: prever sempre a média do treino
previsao_boba = np.full(len(y_teste), y_treino.mean())
mae_bobo = mean_absolute_error(y_teste, previsao_boba)
r2_bobo = r2_score(y_teste, previsao_boba)

# regressão linear
modelo_linear = LinearRegression()
modelo_linear.fit(X_treino, y_treino)
previsoes = modelo_linear.predict(X_teste)
mae_linear = mean_absolute_error(y_teste, previsoes)
r2_linear = r2_score(y_teste, previsoes)

print(f"Modelo bobo:       MAE = {mae_bobo:.4f}   R2 = {r2_bobo:.3f}")
print(f"Regressão linear:  MAE = {mae_linear:.4f}   R2 = {r2_linear:.3f}")

Modelo bobo:       MAE = 0.0445   R2 = -0.000
Regressão linear:  MAE = 0.0440   R2 = 0.012


O que é o "modelo bobo"? Provavelmente é um modelo que sempre prevê a mesma coisa pra todo mundo — geralmente a média (ou mediana) da variável alvo (no seu caso, a taxa_engajamento), sem olhar pra nenhuma feature. Ele serve como "linha de base": se seu modelo de verdade não conseguir ser melhor que isso, ele não está aprendendo nada útil.

MAE (Mean Absolute Error / Erro Absoluto Médio): é, em média, o quanto a previsão do modelo erra pra mais ou pra menos, em relação ao valor real — na mesma unidade da variável que você está prevendo. Quanto menor, melhor.

Modelo bobo: MAE = 0.0445 (erra em média 4,45 pontos percentuais de engajamento)
Regressão linear: MAE = 0.0440 (erra em média 4,40 pontos percentuais)

A diferença é bem pequena: a regressão linear errou só um pouquinho menos que simplesmente "chutar a média pra todo mundo".

R² (coeficiente de determinação): mede o quanto da variação da variável alvo o modelo consegue "explicar" usando as features. Vai de (teoricamente) até 1:

R² = 1 → o modelo explica perfeitamente os dados.
R² = 0 → o modelo é tão bom quanto simplesmente prever a média sempre (ou seja, não está usando as features pra nada de útil).
R² negativo → o modelo é pior que só prever a média.

No seu caso:

Modelo bobo: R² = -0.000 → isso faz todo sentido, porque por definição o modelo "bobo" (que prevê a média) tem R² igual a zero (ou bem próximo, por arredondamento).
Regressão linear: R² = 0.012 → isso significa que as variáveis (features) que você usou explicam só 1,2% da variação da taxa de engajamento. É um valor extremamente baixo.

Interpretação geral: seu modelo de regressão linear está praticamente empatado com o modelo bobo. Ele até é levemente melhor (MAE um pouquinho menor, R² levemente positivo), mas a melhora é tão pequena que, na prática, isso indica que as variáveis que você usou como features não têm relação linear forte com a taxa de engajamento — pelo menos não capturada por um modelo linear simples.

## Parte 4 — Uma árvore, para comparar

Complete a célula: treine uma `DecisionTreeRegressor` (use `max_depth=5, random_state=42`), preveja em `X_teste` e calcule MAE e R². Imprima os três modelos lado a lado (bobo, linear, árvore).

In [17]:
from sklearn.tree import DecisionTreeRegressor

# complete aqui: crie, treine (fit), preveja (predict) e meça (mae/r2) a árvore
modelo_arvore = DecisionTreeRegressor(max_depth=5, random_state=42)
modelo_arvore.fit(X_treino, y_treino)
previsoes_arvore = modelo_arvore.predict(X_teste)

mae_arvore = mean_absolute_error(y_teste, previsoes_arvore)
r2_arvore = r2_score(y_teste, previsoes_arvore)


print(f"Modelo bobo:       MAE = {mae_bobo:.4f}   R2 = {r2_bobo:.3f}")
print(f"Regressão linear:  MAE = {mae_linear:.4f}   R2 = {r2_linear:.3f}")
print(f"Árvore (prof. 5):  MAE = {mae_arvore:.4f}   R2 = {r2_arvore:.3f}")

Modelo bobo:       MAE = 0.0445   R2 = -0.000
Regressão linear:  MAE = 0.0440   R2 = 0.012
Árvore (prof. 5):  MAE = 0.0422   R2 = -0.080


## Parte 5 — Ler os coeficientes

Complete: monte um DataFrame com `X.columns` e `modelo_linear.coef_`, ordenado pelo coeficiente. Olhe o sinal de cada um e escreva, no rascunho abaixo, **uma** leitura em uma frase (lembrando: associação nesta coleta, não causa).

In [18]:
# complete aqui: DataFrame com feature e coeficiente, ordenado
coeficientes = pd.DataFrame({
    "feature": X.columns,
    "coeficiente": modelo_linear.coef_,
}).sort_values("coeficiente")
coeficientes

,feature,coeficiente
2,n_hashtags,-1.313639e-04
1,tam_legenda,-2.714864e-05
0,seguidores_autor,1.919111e-09
3,hora,6.658876e-04


**Rascunho da leitura de um coeficiente (vai para o README):**

> Nesta coleta, posts com mais hashtags e legendas mais longas tenderam a ter taxa de engajamento levemente menor (coeficientes negativos de `n_hashtags` e `tam_legenda`), enquanto posts publicados em horários mais tardios do dia tenderam a ter engajamento levemente maior (coeficiente positivo de `hora`). O número de seguidores do autor teve coeficiente praticamente nulo quando ajustado pela escala da variável, sugerindo pouca relação linear direta com a taxa de engajamento nesta base. É importante notar que essas são associações observadas nos meus dados, não significa, por exemplo, que usar menos hashtags necessariamente aumenta o engajamento de um post. Além disso, como o R² do modelo foi de apenas 0.012, essas variáveis explicam muito pouco da variação real da taxa de engajamento, então essas tendências devem ser interpretadas com cautela.

análise feita com IA para eu entender melhor:

n_hashtags → -0.000131 (negativo): para cada hashtag a mais no post, a taxa de engajamento prevista cai cerca de 0,013 pontos percentuais, mantendo as outras variáveis fixas. É um efeito negativo, mas bem pequeno.

tam_legenda → -0.0000271 (negativo): para cada caractere a mais na legenda, o engajamento cai cerca de 0,0027 pontos percentuais. Também negativo, mas ainda menor que o efeito das hashtags.

seguidores_autor → 0.0000000019 (positivo, mas minúsculo): aqui é onde entra um ponto importante — esse número parece "quase zero" comparado aos outros, mas é porque a escala da variável é completamente diferente. Número de seguidores costuma variar em milhares ou milhões, enquanto hashtags vão de 0 a uns 20-30, por exemplo. Então um coeficiente pequeno multiplicado por um valor grande (tipo 100.000 seguidores) ainda pode gerar um impacto real.

hora → 0.000666 (positivo, o maior valor absoluto aqui): para cada hora mais tarde no dia que o post é publicado, a taxa de engajamento prevista sobe cerca de 0,067 pontos percentuais, mantendo o resto constante.

Ponto crucial pra sua prova: você não pode comparar diretamente esses coeficientes pra dizer "qual variável é mais importante" só olhando o tamanho do número, porque cada uma está numa escala diferente (hashtags: unidades; seguidores: pode ser milhões; hora: 0 a 23; tamanho da legenda: pode ser centenas de caracteres). Um coeficiente "grande" pode simplesmente refletir que a variável tem uma escala pequena, e vice-versa.

Pra comparar a importância real das variáveis de forma justa, o correto seria padronizar (standardize) as features antes de treinar o modelo — por exemplo, com StandardScaler do scikit-learn, deixando todas com média 0 e desvio padrão 1. Só depois disso os coeficientes ficam comparáveis entre si em "força de efeito".

E vale lembrar do contexto: como o R² do seu modelo foi 0.012, todos esses efeitos — mesmo os que parecem "fazer sentido" (postar mais tarde ajuda, muitas hashtags atrapalha) — são estatisticamente bem fracos e explicam muito pouco da variação real do engajamento. Então essa tabela é mais um indício de tendências do que uma conclusão forte.

## Parte 6 — README de reprodução

Crie o arquivo `README.md` dentro de `projetos/11-regressao-scikit-learn/` na sua pasta de entregas, respondendo:

**Fonte e período dos dados:**

> Escreva aqui.

**Quantos posts entraram no modelo (depois de remover duplicata e `plays` = 0):**

> Escreva aqui.

**Quais features você usou, e por quê:**

> Escreva aqui (liste as colunas de `X`).

**Quais colunas você deixou de fora por vazamento, e por quê:**

> Escreva aqui.

**Resultado: MAE e R² do modelo bobo, da linear e da árvore. O seu melhor modelo bateu o bobo?**

> Escreva aqui.

**Uma leitura de coeficiente (associação, não causa):**

> Escreva aqui.

**Declaração de uso de IA:** ferramenta usada, em que trecho ou decisão desta entrega, e o que você conferiu ou alterou depois do resultado gerado (mesmo que a resposta seja "não usei IA nesta entrega", registre isso).

> Escreva aqui.

## Parte 7 — Conferência final

Antes de considerar a entrega concluída, confira:

- [ ] `dados/exportacao.csv` é a sua coleta das Aulas 5 a 7, e a pasta `dados/` está no `.gitignore`.
- [ ] O `X` tem pelo menos três features derivadas por você, todas numéricas, nenhuma delas `likes`/`comments`/`shares`/`plays`.
- [ ] A regressão linear e a árvore foram treinadas e comparadas com o modelo bobo.
- [ ] O README tem: fonte, nº de posts, lista de features, colunas descartadas por vazamento, os três resultados (MAE/R²) e uma leitura de coeficiente.
- [ ] Este notebook roda do início ao fim sem erro com Kernel → Restart e Run All (ou o equivalente no VS Code).
- [ ] O README registra o uso de IA (ou informa que não houve).
- [ ] Este notebook e o README estão copiados dentro de `projetos/11-regressao-scikit-learn/` na sua pasta de entregas.
- [ ] Você já fez `git add`, `git commit` e `git push` dessa entrega.